# 03 — Application-output regression against `Latest_Workplace`

This notebook validates that Yamada results used in the user-guide/application workflows are unchanged by the optimized branch.

It runs the cross-branch output fingerprint script twice:
- once from `Latest_Workplace`;
- once from `perf/yamada-max-optimization`;

and requires a literal JSON equality check. The fingerprint includes crossing counts, PD codes, recursive/Negami results, normalization modes, and public dispatch settings used by the application workflows.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, tempfile

ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/"pyproject.toml").exists():
    ROOT=ROOT.parent
SRC=ROOT/"src"
sys.path.insert(0,str(SRC))
branch=subprocess.check_output(["git","rev-parse","--abbrev-ref","HEAD"],cwd=ROOT,text=True).strip()
print("ROOT =",ROOT)
print("branch =",branch)
if branch!="perf/yamada-max-optimization":
    raise RuntimeError("Run this notebook from perf/yamada-max-optimization.")
import knotted_graph
assert SRC in Path(knotted_graph.__file__).resolve().parents

In [ ]:
script=ROOT/"dev"/"yamada_output_fingerprint.py"
if not script.exists():
    raise FileNotFoundError(script)

def run_ref(ref):
    with tempfile.TemporaryDirectory() as td:
        work=Path(td)/"repo"
        subprocess.run(["git","worktree","add","--detach",str(work),ref],cwd=ROOT,check=True,
                       stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        try:
            env=dict(os.environ)
            env["PYTHONPATH"]=str(work/"src")
            out=subprocess.check_output([sys.executable,str(work/"dev"/"yamada_output_fingerprint.py")],
                                        cwd=work,env=env,text=True)
            return json.loads(out)
        finally:
            subprocess.run(["git","worktree","remove","--force",str(work)],cwd=ROOT,check=False,
                           stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)

baseline=run_ref("Latest_Workplace")
optimized=run_ref("perf/yamada-max-optimization")
assert baseline==optimized, "Application/end-to-end Yamada fingerprint changed!"
print("PASS: optimized branch is byte-identical to Latest_Workplace for every fingerprinted case.")
print("cases =",len(baseline) if hasattr(baseline,"__len__") else "n/a")

## Notebook-source guarantee

All main user-guide notebooks locate the repository root and prepend `<checkout>/src` before importing `knotted_graph`. Therefore running them from your local `KnottedGraph-perf-yamada-max-optimization` checkout uses the optimized branch source, not a stale pip installation.

The setup cells in Getting Started, Core Workflows, Advanced/Reproduction, Physics Applications and Mathematics Applications already use this pattern.